# Experiment: teaching a 0.6B model the answer contract with LoRA

**This is an experiment, not part of the shipped pipeline.** The pipeline answers with SmolLM3 3B on llama.cpp; nothing here changes it.

**Question.** The pipeline's prompt asks for an answer contract: cite each source as `[n]`, and when the sources don't hold the answer, reply with exactly *I cannot answer this from the provided documents.* Can a LoRA adapter, trained on a few hundred examples generated automatically from the corpus, teach that contract to a model too small to follow it from the prompt alone? And does the gain carry over to the eval questions, none of which it trained on?

**Held fixed:** retrieval (hybrid search, top 5), the prompt (`grounded-v2`), the graders and the three eval sets. Only the model changes: [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) (Apache 2.0) without and with the adapter, greedy decoding, reasoning mode off.

**How the training data avoids the eval** (details in `bankrag/finetune_data.py`, built by `python -m bankrag finetune-data`):
1. Only chunks from articles that no eval question is about.
2. SmolLM3 writes one question, answer and supporting quote per chunk; code drops it unless the quote is in the chunk word for word and the question stands on its own.
3. Each question becomes an *answer* example (its chunk is among the five retrieved sources; target = answer + `[n]`) and, for 60% of questions, a *refuse* example (every chunk of that article removed from the sources; target = the refusal sentence). Same question, with and without the evidence, so the model has to look at the sources.

In [ ]:
import json
import math
import random
import sys
import time
from collections import Counter
from pathlib import Path

if Path.cwd().name == "notebooks":
    sys.path.insert(0, str(Path.cwd().parent))

import peft
import torch
import transformers
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

from bankrag import finetune_data as fd
from bankrag import hf_chat
from bankrag.config import LORA_ADAPTER_DIR
from bankrag.evaluate import run_eval
from bankrag.grading import load_questions, normalise
from bankrag.index import build_index
from bankrag.report import latest_runs

MODEL_ID = "Qwen/Qwen3-0.6B"
PROMPT = "grounded-v2"
SETS = ["main", "holdout", "plain"]
print(f"torch {torch.__version__} (CUDA {torch.version.cuda}), transformers {transformers.__version__}, peft {peft.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

## 1. Training data

In [ ]:
generated = fd.load_jsonl(fd.GENERATED_FILE)
examples = fd.load_jsonl(fd.EXAMPLES_FILE)
print(f"{len(generated)} chunks shown to the generator, from {len({g['article_id'] for g in generated})} articles")
for reason, n in Counter(g["rejected"] or "kept" for g in generated).most_common():
    print(f"  {n:4}  {reason}")
print()
for (split, kind), n in sorted(Counter((e["split"], e["kind"]) for e in examples).items()):
    print(f"  {split:5} {kind:6} {n}")

# Leak checks: no training example comes from an eval question's gold article, and no
# training question repeats an eval question.
gold = fd.eval_gold_articles()
sources = {e["id"].rsplit("-", 2)[0] for e in examples}
assert not sources & gold, sources & gold
eval_questions = {normalise(q.question) for s in SETS for q in load_questions(s)}
assert not eval_questions & {normalise(e["question"]) for e in examples}
print(f"\nleak checks passed: {len(sources)} source articles, none of the {len(gold)} eval gold articles")

In [ ]:
def words(text):
    return set(normalise(text).replace("?", "").split())

# The training questions most similar to an eval question, by word overlap (Jaccard).
pairs = []
for q in (q for s in SETS for q in load_questions(s)):
    best = max(examples, key=lambda e: len(words(e["question"]) & words(q.question)) / len(words(e["question"]) | words(q.question)))
    sim = len(words(best["question"]) & words(q.question)) / len(words(best["question"]) | words(q.question))
    pairs.append((sim, q.question, best["question"]))
for sim, q, t in sorted(pairs, reverse=True)[:3]:
    print(f"{sim:.2f}  eval:  {q}\n      train: {t}\n")

In [ ]:
chat = fd.to_chat(examples)
for kind in ("answer", "refuse"):
    ex = next(c for c in chat if c.kind == kind)
    print(f"--- {kind} example: {ex.id}")
    print(ex.user[-700:])
    print(f"TARGET: {ex.target}\n")

## 2. The base model on the eval sets

In [ ]:
index = build_index()
base = hf_chat.load(MODEL_ID)
runs = {}
for s in SETS:
    runs[("base", s)] = run_eval("qwen3-0.6b", PROMPT, index=index, question_set=s, llm=base, progress=False)
    print(f"base {s:8} answer accuracy {runs[('base', s)].summary['answer_accuracy']:.3f}")
del base
torch.cuda.empty_cache()

## 3. LoRA training

Rank 16 on every attention and MLP projection, 2 epochs, learning rate 2e-4 with cosine decay, an effective batch of 8 examples. The loss counts only the answer tokens, not the prompt. The prompt tokens are made by the same function the evaluation uses, so the adapter is trained on exactly the prompt it will see.

The model scores its whole 152,000-token vocabulary at every position, and a prompt here is about 1,100 tokens long, so computing those scores for every position takes several GiB. Only the answer positions are needed for the loss, so `logits_to_keep` asks for just those. That gives the same loss (checked on one example: 0.7906 both ways) and fits the 8 GB card.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
END = tok("<|im_end|>\n", add_special_tokens=False)["input_ids"]  # Qwen's end of turn


def encode(ex):
    prompt = hf_chat.chat_prompt_ids(tok, ex.system, ex.user)
    target = tok(ex.target, add_special_tokens=False)["input_ids"] + END
    return {"input_ids": prompt + target, "prompt_len": len(prompt), "id": ex.id}


train = [encode(c) for c in chat if c.split == "train"]
val = [encode(c) for c in chat if c.split == "val"]
lengths = sorted(len(e["input_ids"]) for e in train)
print(f"{len(train)} training and {len(val)} validation examples; "
      f"length median {lengths[len(lengths) // 2]}, max {lengths[-1]} tokens")

In [ ]:
random.seed(0)
torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda")
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
EPOCHS, LR, ACCUM = 2, 2e-4, 8


def answer_loss(ex):
    # Mean cross-entropy over the answer tokens, with scores computed only where they are needed.
    ids = torch.tensor([ex["input_ids"]], device="cuda")
    p = ex["prompt_len"]
    logits = model(input_ids=ids, logits_to_keep=ids.shape[1] - p + 1).logits[0, :-1].float()
    return torch.nn.functional.cross_entropy(logits, ids[0, p:])


def mean_loss(data):
    model.eval()
    with torch.no_grad():
        total = sum(answer_loss(ex).item() for ex in data)
    model.train()
    return total / len(data)


params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
steps = math.ceil(len(train) / ACCUM) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, steps // 20), steps)
torch.cuda.reset_peak_memory_stats()
history = [{"epoch": 0, "step": 0, "train_loss": None, "val_loss": mean_loss(val)}]
print(f"before training: validation loss {history[0]['val_loss']:.3f}")

t0, step = time.perf_counter(), 0
model.train()
for epoch in range(1, EPOCHS + 1):
    order = list(range(len(train)))
    random.Random(epoch).shuffle(order)
    running, seen = 0.0, 0
    for i, j in enumerate(order, 1):
        loss = answer_loss(train[j])
        (loss / ACCUM).backward()
        running, seen = running + loss.item(), seen + 1
        if i % ACCUM == 0 or i == len(order):
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step += 1
    history.append({"epoch": epoch, "step": step, "train_loss": running / seen, "val_loss": mean_loss(val)})
    print(f"epoch {epoch}: train loss {running / seen:.3f}, validation loss {history[-1]['val_loss']:.3f}, "
          f"{time.perf_counter() - t0:.0f} s")

print(f"\n{step} optimizer steps in {time.perf_counter() - t0:.0f} s, "
      f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
model.save_pretrained(LORA_ADAPTER_DIR)
RESULTS = Path(fd.__file__).resolve().parent.parent / "results" / "lora"
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "training.json").write_text(json.dumps({
    "model": MODEL_ID, "lora": {"r": lora.r, "alpha": lora.lora_alpha, "dropout": lora.lora_dropout,
                                "target_modules": sorted(lora.target_modules)},
    "epochs": EPOCHS, "learning_rate": LR, "grad_accum": ACCUM,
    "train_examples": len(train), "val_examples": len(val), "optimizer_steps": step,
    "seconds": round(time.perf_counter() - t0, 1),
    "peak_gpu_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2), "history": history,
}, indent=2), encoding="utf-8")
print(f"adapter saved to {LORA_ADAPTER_DIR.relative_to(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())}")

## 4. The model with the adapter on the eval sets

In [ ]:
# Fold the adapter into the weights, so answering is as fast as with the base model.
merged = model.merge_and_unload()
merged.gradient_checkpointing_disable()
merged.config.use_cache = True
merged.eval()
tuned = hf_chat.HFChatModel(model=merged, tokenizer=tok)
for s in SETS:
    runs[("lora", s)] = run_eval("qwen3-0.6b-lora", PROMPT, index=index, question_set=s, llm=tuned, progress=False)
    print(f"lora {s:8} answer accuracy {runs[('lora', s)].summary['answer_accuracy']:.3f}")

In [ ]:
def cited_share(run):
    """Of the answers that aren't refusals, the share that cite at least one valid source."""
    answered = [r for r in run.rows if not r["refused"]]
    return sum(r["citation_valid"] for r in answered) / len(answered) if answered else float("nan")


smol = latest_runs()
rows = []
for s in SETS:
    for name, summary, cited in (
        ("Qwen3 0.6B", runs[("base", s)].summary, cited_share(runs[("base", s)])),
        ("Qwen3 0.6B + LoRA", runs[("lora", s)].summary, cited_share(runs[("lora", s)])),
        ("SmolLM3 3B (reference)", smol[("local", PROMPT, s)]["summary"], None),
    ):
        refusal = f"{summary['refusal_rate']:.0%}" if summary["n_unanswerable"] else "-"
        rows.append(
            f"| {s} | {name} | {summary['answer_accuracy']:.0%} | {summary['citation_accuracy']:.0%} | {refusal} "
            f"| {summary['false_refusal_rate']:.0%} | {'-' if cited is None else f'{cited:.0%}'} "
            f"| {summary['tokens_out_mean']:.0f} | {summary['latency_ms_p50'] / 1000:.2f} s |"
        )
print("| Set | Model | Answer accuracy | Cites the right article | Refuses when it should | Wrongly refuses "
      "| Answers with a valid citation | Tokens out | Median latency |")
print("|---|---|---|---|---|---|---|---|---|")
print("\n".join(rows))

In [ ]:
by_kind = {}
for who in ("base", "lora"):
    for s in SETS:
        for kind, v in runs[(who, s)].summary["by_kind"].items():
            c = by_kind.setdefault((who, kind), [0, 0])
            c[0] += v["correct"]
            c[1] += v["n"]
for kind in ("fact", "false_premise", "out_of_corpus", "injection"):
    b, l = by_kind.get(("base", kind), [0, 0]), by_kind.get(("lora", kind), [0, 0])
    print(f"{kind:14} base {b[0]:2}/{b[1]:<2}  lora {l[0]:2}/{l[1]:<2}")

In [ ]:
# The same questions, answered by both models.
base_rows = {r["id"]: r for s in SETS for r in runs[("base", s)].rows}
lora_rows = {r["id"]: r for s in SETS for r in runs[("lora", s)].rows}
for qid in ["psd2-liability-cap", "out-cet1-ratio", "inject-ignore-rules", "p-stolen-card-400"]:
    b, l = base_rows[qid], lora_rows[qid]
    print(f"=== {qid}: {b['question']}")
    print(f"  base ({'ok' if b['correct'] else 'wrong'}): {b['answer'][:300]}")
    print(f"  lora ({'ok' if l['correct'] else 'wrong'}): {l['answer'][:300]}\n")

## Findings

*(Written after the run above.)*